# Practical 08 — Domain-Specific Financial LLMs

This notebook runs the chapter's general-vs-domain comparison **directly through the
reference tools** in `tools/`, fully offline. The pipeline scores the same labelled
finance-sentiment dataset twice — once with a naive general-English polarity lexicon
(standing in for a web-trained model) and once with a finance-adapted lexicon
(standing in for a FinBERT-style domain model) — then adjudicates which classifier
wins and *why*, naming the exact finance terms (`beat`, `headwinds`, `impairment`,
`covenant`) that flip each verdict.

In the actual project you drive these tools agentically from Claude Code / Cline via
the `/compare` command and the `generalist` / `specialist` / `adjudicator` sub-agents.
Colab can't run that loop, so here we call the same deterministic tool functions
ourselves and watch the whole pipeline produce its real outputs.

In [ ]:
# This practical's tools are Python standard library only —
# nothing to install. (Colab already has Python ready to go.)

In [ ]:
# --- setup: make this practical's tools importable (works locally & in Colab) ---
import os, sys, pathlib, subprocess
PRACTICAL = "08-domain-specific-llms"
def _locate():
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if base.name == PRACTICAL and (base / "tools").is_dir():
            return base
        cand = base / "code" / "practicals" / PRACTICAL
        if cand.is_dir():
            return cand
    return None
root = _locate()
if root is None:
    if not pathlib.Path("llm-finance-book").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/jfimbett/llm-finance-book.git"], check=True)
    root = pathlib.Path("llm-finance-book/code/practicals") / PRACTICAL
root = root.resolve()
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "tools"))
os.chdir(root)
print("Practical root:", root)

## Step 0 — The bundled data

Everything runs offline from three files under `data/`:

- `sentences.jsonl` — a small fictional labelled finance-sentiment dataset
  (`{id, text, label}` per line, label ∈ `positive` / `negative` / `neutral`).
- `lexicon_general.json` — a naive English polarity lexicon (`{word: ±1}`). It scores
  `beat` as negative (a beating) and `covenant` as positive, and has never seen
  `headwinds`, `impairment`, or `accretive`.
- `lexicon_domain.json` — the finance-adapted lexicon, where those same terms are
  scored the way finance actually uses them.

Let's load them with the tools' own helpers and look at what we're working with.

In [ ]:
import json
from tools._common import load_sentences, load_lexicon

rows = load_sentences()
gen_lex = load_lexicon("general")
dom_lex = load_lexicon("domain")

print(f"{len(rows)} labelled sentences   "
      f"| general lexicon: {len(gen_lex)} words   "
      f"| domain lexicon: {len(dom_lex)} words\n")
for r in rows[:5]:
    print(f"  [{r['id']}] {r['label']:8s}  {r['text']}")
print("  ...")

## How a classifier works here

Each "classifier" is just a lexicon plus a rule: tokenise the sentence into bare
lowercase words, sum the polarity of every token found in the lexicon (unknown tokens
contribute 0), and take the **sign** of the sum as the label. A missing finance term
is therefore silently dropped — exactly how a model that never learned the word behaves.

We can see the mechanism on one sentence with `score_sentence` / `classify`.

In [ ]:
from tools._common import tokenize, score_sentence, classify

probe = "The company beat consensus estimates this quarter."
print("tokens:", tokenize(probe), "\n")
print(f"general: score={score_sentence(probe, gen_lex):+d}  ->  {classify(probe, gen_lex)}")
print(f"domain : score={score_sentence(probe, dom_lex):+d}  ->  {classify(probe, dom_lex)}")
print("\nGold label for this sentence:", rows[0]["label"])
print("The single word 'beat' is scored -1 (general) vs +1 (domain) — that flips the verdict.")

## Step 1 — Score the general-purpose baseline

`tools.general` is the general-English classifier. `predict_all` returns one label per
sentence; we score it against the gold labels with `accuracy`. This is what
`python -m tools.general` prints, reconstructed here so we can see every per-sentence
hit and miss.

In [ ]:
from tools import general
from tools._common import accuracy

gold = [r["label"] for r in rows]
gen_pred = general.predict_all(rows)
gen_acc = accuracy(gen_pred, gold)

print(f"general-purpose accuracy: {gen_acc:.3f}\n")
print(f"  {'id':4s} {'gold':9s} {'pred':9s} ok")
for r, p in zip(rows, gen_pred):
    print(f"  {r['id']:4s} {r['label']:9s} {p:9s} {'OK' if p == r['label'] else 'X'}")

## Step 2 — Score the finance-domain model

`tools.domain` uses the finance-adapted lexicon. Same code path, different lexicon.

In [ ]:
from tools import domain

dom_pred = domain.predict_all(rows)
dom_acc = accuracy(dom_pred, gold)

print(f"finance-domain accuracy: {dom_acc:.3f}\n")
print(f"  {'id':4s} {'gold':9s} {'pred':9s} ok")
for r, p in zip(rows, dom_pred):
    print(f"  {r['id']:4s} {r['label']:9s} {p:9s} {'OK' if p == r['label'] else 'X'}")

## Step 3 — Adjudicate: accuracies, winner, and the deciding terms

`tools.compare.compare()` runs both classifiers, scores them, names the winner and the
margin, and — crucially — lists every sentence the domain model got right that the
general model missed, annotated with the `deciding_terms`: tokens whose general vs
domain polarity disagree (including terms the general lexicon never saw, shown as
`absent`). This is the structured object the `adjudicator` agent reads.

In [ ]:
from tools.compare import compare

result = compare()
summary = {k: result[k] for k in
           ("n_sentences", "general_accuracy", "domain_accuracy", "winner", "margin")}
print(json.dumps(summary, indent=2))
print(f"\nDomain won {len(result['domain_wins'])} sentences the general model missed.")

### Per-sentence: where domain adaptation paid off

Each flipped sentence below shows the gold label, both predictions, and the exact
finance terms responsible. Notice the recurring offenders the chapter calls out:
`beat` (general −1 → domain +1), `headwinds` / `impairment` / `accretive` (unseen by
the general lexicon), and `liability` / `covenant` (loaded in general English,
neutral on a balance sheet).

In [ ]:
for w in result["domain_wins"]:
    terms = ", ".join(
        f"{t['term']}(gen={t['general']}, dom={t['domain']})"
        for t in w["deciding_terms"]
    )
    print(f"[{w['id']}] gold={w['gold']:8s} general={w['general_pred']:8s} domain={w['domain_pred']:8s}")
    print(f'      "{w["text"]}"')
    print(f"      deciding terms: {terms}\n")

## Step 4 — The human-readable verdict (what gets saved to reports/)

`tools.compare._format` renders the same result as the text block the agent would
write to `reports/compare.md`. We print it rather than write a file, to keep the
notebook side-effect-free.

In [ ]:
from tools.compare import _format

print(_format(result))

## Try it / interpret — "domain adaptation, done by hand"

The README suggests: classify the same sentence with each model and watch them disagree
on a single word. Below we confirm `beat` is the pivot, then *manually adapt* the
general lexicon by teaching it the missing finance terms — and watch its accuracy climb
toward the domain model. This is exactly what domain adaptation buys you, made concrete.

In [ ]:
sent = "The company beat consensus estimates."
print("single-sentence disagreement:")
print(f"  general -> {general.classify_sentence(sent)}")
print(f"  domain  -> {domain.classify_sentence(sent)}")
print("  (they differ on one word: 'beat')\n")

# Patch a COPY of the general lexicon with the finance terms it was missing,
# then re-score with the shared classify() rule (no global state mutated).
patched = dict(gen_lex)
patched.update({"beat": 1, "headwinds": -1, "headwind": -1,
                "impairment": -1, "accretive": 1, "tailwinds": 1})
patched_pred = [classify(r["text"], patched) for r in rows]
patched_acc = accuracy(patched_pred, gold)

print(f"general accuracy (original) : {gen_acc:.3f}")
print(f"general accuracy (patched)  : {patched_acc:.3f}")
print(f"finance-domain accuracy     : {dom_acc:.3f}")
print("\nTeaching the general lexicon a handful of finance terms recovers most of the gap —")
print("which is the chapter's point: the domain win comes from finance vocabulary, not magic.")

## Recap / next steps

We ran the practical's reference pipeline end to end, offline:

1. **Baseline** — scored the general-English classifier (`tools.general`): ~0.29 accuracy.
2. **Domain** — scored the finance-adapted classifier (`tools.domain`): 1.00 accuracy.
3. **Adjudicate** — `tools.compare.compare()` named the winner (**domain**, margin ≈ 0.71)
   and listed every flipped sentence with its `deciding_terms`.
4. **Explain / try it** — confirmed the verdict turns on specific finance words, then
   hand-adapted the general lexicon to close most of the gap.

Next steps from the README's "things to try":

- Edit `data/lexicon_general.json` on disk (add `headwind`, `impairment`, `accretive`)
  and re-run — the change persists for the CLI tools too.
- Add a new labelled line to `data/sentences.jsonl` whose verdict hinges on a finance
  term, then re-run `compare()`.
- Run the offline test suite with `pytest` to see the comparison gate
  (`domain_accuracy > general_accuracy`) enforced.
- In the full project, drive all of this agentically via `/compare` in Claude Code / Cline.